# 15 · Export, compression and deployment parity

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Exports a visual predictor for research, not a complete medically validated app. Rights and the source/selection engine remain separate.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Export the selected registered visual model

In [ ]:
from oncoplate.export import export_run,benchmark_export
RUN_ID='resnet50_frozen_joint_s0'
DO_ONNX=False  # Install -e .[export] before enabling; TorchScript route works without ONNX.
contract=export_run(p['runs']/RUN_ID,p['exports']/RUN_ID,temperature=1.,onnx=DO_ONNX)
print(json.dumps(contract,indent=2))

## 2. Compare outputs on real validation images, not only a zero tensor

In [ ]:
from oncoplate.pipeline import load_study
from oncoplate.export import image_model_for_run,ExportModel
from oncoplate.vision import Letterbox
from PIL import Image
import torch,numpy as np
records,targets=load_study(cfg,'joint',stage_images=True)
subset=records[records.split.eq('validation')].head(32)
x=torch.stack([Letterbox(contract['image_size'])(Image.open(path)) for path in subset.image_path])
model,_,_=image_model_for_run(p['runs']/RUN_ID)
with torch.inference_mode():expected=ExportModel(model).eval()(x).numpy()
traced=torch.jit.load(str(p['exports']/RUN_ID/'predictor.torchscript.pt')).eval()
with torch.inference_mode():actual=traced(x).numpy()
parity={'records':len(subset),'maximum_absolute_difference':float(np.max(np.abs(expected-actual))), 'scope':'visual_prediction_only','real_cases':True}
write_json(p['exports']/RUN_ID/'validation_export_parity.json',parity);print(parity)

## 3. Runtime latency and head-only quantisation ablation

In [ ]:
timing=benchmark_export(p['exports']/RUN_ID/'predictor.torchscript.pt',x[:1].numpy(),repeats=30)
print({k:v for k,v in timing.items() if k!='outputs'})
write_json(p['exports']/RUN_ID/'host_latency.json',{k:v for k,v in timing.items() if k!='outputs'})
print('Host latency is not phone latency. Device battery/thermal testing must be measured on the actual devices.')

## 4. Inspect prototype and rights requirements

In [ ]:
print((REPO/'app/README.md').read_text())
print((REPO/'docs/MOBILE_AND_RIGHTS.md').read_text())

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
